In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_image_batched_inference.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    python_exe = sys.executable
    !{python_exe} -m pip install opencv-python matplotlib scikit-learn huggingface_hub
    !{python_exe} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [ ]:
from pathlib import Path
import os

def _load_export_style_env_file(env_path: Path):
    loaded_keys = []
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value
        loaded_keys.append(key)
    return loaded_keys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / ".env").exists():
    repo_root = repo_root.parent

env_file = repo_root / ".env"
if env_file.exists():
    loaded_env_keys = _load_export_style_env_file(env_file)
    print(f"Loaded {len(loaded_env_keys)} vars from {env_file}")
else:
    loaded_env_keys = []
    print("No .env file found while walking up from current working directory")

hf_home = os.environ.get("HF_HOME", "").strip()
if hf_home:
    os.makedirs(hf_home, exist_ok=True)
    if "HF_HUB_CACHE" not in os.environ:
        os.environ["HF_HUB_CACHE"] = hf_home if hf_home.endswith("/hub") else f"{hf_home}/hub"

hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("Hugging Face authentication configured from .env")
    except Exception as exc:
        print(f"Hugging Face login skipped: {exc}")

from PIL import Image
import requests
from io import BytesIO
import sam3

sam3_root = os.path.abspath(os.path.join(os.path.dirname(sam3.__file__), ".."))
print(f"SAM3 root: {sam3_root}")
print(f"HF_HOME: {os.environ.get('HF_HOME', '<default>')}")

In [ ]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    compute_device = torch.device("cuda")
else:
    compute_device = torch.device("cpu")

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()
print(f"Compute device: {compute_device}")

# Utilities

## Plotting

This section contains simple utilities to plot masks and bounding masks on top of an image

In [ ]:
# Plot utility from sam3.visualization_utils is not needed for prompt-free mode.
# We render masks with matplotlib in the plotting section below.

## Batching

This section contains some utility functions to create datapoints. They are optional, but give some good indication on how they should be created

In [ ]:
from sam3.train.data.sam3_image_dataset import InferenceMetadata, FindQueryLoaded, Image as SAMImage, Datapoint
from typing import List

GLOBAL_COUNTER = 1
def create_empty_datapoint():
    """ A datapoint is a single image on which we can apply several queries at once. """
    return Datapoint(find_queries=[], images=[])

def set_image(datapoint, pil_image):
    """ Add the image to be processed to the datapoint """
    w,h = pil_image.size
    datapoint.images = [SAMImage(data=pil_image, objects=[], size=[h,w])]

def add_text_prompt(datapoint, text_query):
    """ Add a text query to the datapoint """

    global GLOBAL_COUNTER
    # in this function, we require that the image is already set.
    # that's because we'll get its size to figure out what dimension to resize masks and boxes
    # In practice you're free to set any size you want, just edit the rest of the function
    assert len(datapoint.images) == 1, "please set the image first"

    w, h = datapoint.images[0].size
    datapoint.find_queries.append(
        FindQueryLoaded(
            query_text=text_query,
            image_id=0,
            object_ids_output=[], # unused for inference
            is_exhaustive=True, # unused for inference
            query_processing_order=0,
            inference_metadata=InferenceMetadata(
                coco_image_id=GLOBAL_COUNTER,
                original_image_id=GLOBAL_COUNTER,
                original_category_id=1,
                original_size=[w, h],
                object_id=0,
                frame_index=0,
            )
        )
    )
    GLOBAL_COUNTER += 1
    return GLOBAL_COUNTER - 1

def add_visual_prompt(datapoint, boxes:List[List[float]], labels:List[bool], text_prompt="visual"):
    """ Add a visual query to the datapoint.
    The bboxes are expected in XYXY format (top left and bottom right corners)
    For each bbox, we expect a label (true or false). The model tries to find boxes that ressemble the positive ones while avoiding the negative ones
    We can also give a text_prompt as an additional hint. It's not mandatory, leave it to "visual" if you want the model to solely rely on the boxes.

    Note that the model expects the prompt to be consistent. If the text reads "elephant" but the provided boxe points to a dog, the results will be undefined.
    """

    global GLOBAL_COUNTER
    # in this function, we require that the image is already set.
    # that's because we'll get its size to figure out what dimension to resize masks and boxes
    # In practice you're free to set any size you want, just edit the rest of the function
    assert len(datapoint.images) == 1, "please set the image first"
    assert len(boxes) > 0, "please provide at least one box"
    assert len(boxes) == len(labels), f"Expecting one label per box. Found {len(boxes)} boxes but {len(labels)} labels"
    for b in boxes:
        assert len(b) == 4, f"Boxes must have 4 coordinates, found {len(b)}"

    labels = torch.tensor(labels, dtype=torch.bool).view(-1)
    if not labels.any().item() and text_prompt=="visual":
        print("Warning: you provided no positive box, nor any text prompt. The prompt is ambiguous and the results will be undefined")
    w, h = datapoint.images[0].size
    datapoint.find_queries.append(
        FindQueryLoaded(
            query_text=text_prompt,
            image_id=0,
            object_ids_output=[], # unused for inference
            is_exhaustive=True, # unused for inference
            query_processing_order=0,
            input_bbox=torch.tensor(boxes, dtype=torch.float).view(-1,4),
            input_bbox_label=labels,
            inference_metadata=InferenceMetadata(
                coco_image_id=GLOBAL_COUNTER,
                original_image_id=GLOBAL_COUNTER,
                original_category_id=1,
                original_size=[w, h],
                object_id=0,
                frame_index=0,
            )
        )
    )
    GLOBAL_COUNTER += 1
    return GLOBAL_COUNTER - 1

# Loading

First we load our model

In [ ]:
import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.sam3_utils import load_sam3

sam3_model_dict = load_sam3(
    device=str(compute_device),
    confidence_threshold=0.7,
 )
model = sam3_model_dict["model"].to(compute_device)
model.eval()

print(f"SAM3 model loaded on {compute_device}")
print(f"Batch APIs available: {sam3_model_dict.get('batch_supported', False)}")

Then our validation transforms

In [ ]:
transform = sam3_model_dict.get("transform")
if transform is None:
    print("Wrapper did not return a transform (SAM3 batch APIs unavailable in this install).")
else:
    print("Using transform from src.sam3_utils.load_sam3")

And finally our postprocessor

In [ ]:
postprocessor = sam3_model_dict.get("postprocessor")
if postprocessor is None:
    print("Wrapper did not return a postprocessor (SAM3 batch APIs unavailable in this install).")
else:
    print("Using postprocessor from src.sam3_utils.load_sam3")

# Inference

For inference, we proceed as follows:
- Create each datapoint one by one, using the functions above. Each query that we make will give us a unique id, which is then used after post-processing to retrieve the results
- Each datapoint must be transformed according to are pre-processing transforms (basically resize to 1008x1008, normalize)
- We then collate all datapoints into a batch and forward it to the model

In [ ]:
# Prompt-free setup: load images only (no text or visual prompts)
img1 = Image.open("/mnt/abka03/xlvlm_data/imagenet_3_class/train/cat/n02123045_10633.JPEG").convert("RGB")
img2 = Image.open("/mnt/abka03/xlvlm_data/imagenet_3_class/train/rabbit/n02325366_6190.JPEG").convert("RGB")
print(f"Loaded image 1 size: {img1.size}")
print(f"Loaded image 2 size: {img2.size}")

In [ ]:
# Optional local image example
img_local_example = None
# img_local_example = Image.open(f"{sam3_root}/assets/images/test_image.jpg").convert("RGB")

images_for_auto = [img1, img2]
if img_local_example is not None:
    images_for_auto.append(img_local_example)

masks_per_image = 5
min_mask_area = 100
points_per_side = 10
auto_topn = max(masks_per_image * 2, 20)

print(f"Running prompt-free segmentation on {len(images_for_auto)} image(s)")
print({
    "masks_per_image": masks_per_image,
    "auto_topn": auto_topn,
    "min_mask_area": min_mask_area,
    "points_per_side": points_per_side,
})

In [ ]:
# Prompt-free SAM3 segmentation (point-grid, no text prompt)
from src.sam3_utils import predict_auto_masks_sam3

auto_pairs_per_img = predict_auto_masks_sam3(
    sam3_model_dict,
    images_for_auto,
    topn=auto_topn,
    min_mask_area=min_mask_area,
    points_per_side=points_per_side,
 )

for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    print(f"Image {image_idx} masks: {len(pairs)}")

In [ ]:
# Quick summary of largest masks by area (per image)
def _top_mask_areas(pairs, k=10):
    areas = [int(mask.sum()) for _, mask in pairs if mask is not None]
    areas.sort(reverse=True)
    return areas[:k]

for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    print(f"Image {image_idx} top mask areas:", _top_mask_areas(pairs))

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


# ─── Helper functions ────────────────────────────────────────────────────────

def _mask_overlay(mask, color, alpha=0.45):
    """Create RGBA overlay for mask visualization."""
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., :3] = color
    overlay[..., 3] = mask.astype(np.float32) * alpha
    return overlay


def _ensure_mask_size(mask, image_hw):
    """Ensure mask is bool and matches image (H, W)."""
    h, w = image_hw
    mask = np.asarray(mask).astype(bool)
    if mask.shape != (h, w):
        mask_img = Image.fromarray((mask.astype(np.uint8) * 255))
        mask = np.array(mask_img.resize((w, h), resample=Image.NEAREST)) > 127
    return mask


def _tight_bbox_from_mask(mask):
    """Return tight bbox as (x1, y1, x2, y2), with x2/y2 exclusive."""
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    return x1, y1, x2, y2


def _mask_iou(mask_a, mask_b):
    """Compute IoU between two boolean masks of the same shape."""
    inter = int(np.logical_and(mask_a, mask_b).sum())
    union = int(np.logical_or(mask_a, mask_b).sum())
    if union == 0:
        return 0.0
    return float(inter) / float(union)


def _add_margin(bbox_xyxy, image_hw, margin_frac=0.05, min_margin_px=2):
    """
    Add a small context margin around a tight bounding box.
    Clamp to image boundaries.
    """
    x1, y1, x2, y2 = bbox_xyxy
    h, w = image_hw
    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    mx = max(min_margin_px, int(round(bw * margin_frac)))
    my = max(min_margin_px, int(round(bh * margin_frac)))
    return (
        max(0, x1 - mx),
        max(0, y1 - my),
        min(w, x2 + mx),
        min(h, y2 + my),
    )


def _resize_aspect_pad(img_np, target_hw, pad_value=255):
    """
    Resize img_np (H,W,C) into target_hw (th, tw) preserving aspect ratio.
    Shorter side is padded with pad_value (white).
    The image is centered in the target canvas.
    """
    th, tw = target_hw
    src_h, src_w = img_np.shape[:2]
    if src_h <= 0 or src_w <= 0:
        canvas = np.full((th, tw, 3), pad_value, dtype=np.uint8)
        return canvas

    scale = min(tw / float(src_w), th / float(src_h))
    new_w = max(1, int(round(src_w * scale)))
    new_h = max(1, int(round(src_h * scale)))

    lanczos = getattr(getattr(Image, "Resampling", Image), "LANCZOS", Image.BICUBIC)
    resized = np.array(
        Image.fromarray(img_np).resize((new_w, new_h), resample=lanczos)
    )

    canvas = np.full((th, tw, 3), pad_value, dtype=np.uint8)
    y_off = (th - new_h) // 2
    x_off = (tw - new_w) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas


# ─── Main pipeline ───────────────────────────────────────────────────────────

def build_segment_patches(
    image,
    pairs,
    patch_shape=(200, 200),
    iou_dedup_threshold=0.80,
    bg_color=(255, 255, 255),
    margin_frac=0.05,
    min_margin_px=2,
    min_mask_pixels=25,
):
    """
    Create clean, non-overlapping concept patches from segmentation masks.

    Algorithm
    ---------
    1. Preprocess masks — convert to bool, skip None/empty/tiny.
    2. Dedup by mask IoU — sort by area descending; skip if IoU >= threshold
       with any already-kept mask (same physical segment).
    3. Masked crop — tight bbox + small margin; set non-segment pixels to
       bg_color (white).
    4. Aspect-preserving resize — fit into patch_shape with white padding
       so segments are not distorted.

    Key properties
    --------------
    - Each patch shows ONLY its segment (white background elsewhere).
    - No overlapping image content between patches of different segments.
    - Both parent (cat body) and child (cat ear) segments are kept.
    - Aspect ratio is preserved (no stretching).
    - Dedup uses actual mask pixel IoU only (simple, robust).

    Parameters
    ----------
    image : PIL.Image
    pairs : list of (bbox_xywh, mask_bool_HW) tuples from SAM3
    patch_shape : (H, W) output size
    iou_dedup_threshold : masks with IoU >= this are considered duplicates
    bg_color : RGB tuple for non-segment pixels (default white)
    margin_frac : fractional margin around tight bbox (default 5%)
    min_margin_px : minimum margin in pixels (default 2)
    min_mask_pixels : skip masks smaller than this (noise)

    Returns
    -------
    list of dict with keys:
        mask_index, bbox_xywh, tight_bbox, mask_area, margin_bbox,
        segment_coverage, patch (numpy H,W,3 uint8)
    """
    if patch_shape[0] <= 0 or patch_shape[1] <= 0:
        raise ValueError("patch_shape must be positive")

    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    patch_h, patch_w = int(patch_shape[0]), int(patch_shape[1])
    bg = np.array(bg_color, dtype=np.uint8)

    # ── Step 1: Preprocess ───────────────────────────────────────────────
    valid = []
    for mask_idx, (bbox_xywh, mask) in enumerate(pairs, start=1):
        if mask is None:
            continue
        mask_bool = _ensure_mask_size(mask, (h, w))
        mask_area = int(mask_bool.sum())
        if mask_area < min_mask_pixels:
            continue
        tight_box = _tight_bbox_from_mask(mask_bool)
        if tight_box is None:
            continue
        valid.append({
            "mask_index": mask_idx,
            "bbox_xywh": bbox_xywh,
            "mask_bool": mask_bool,
            "mask_area": mask_area,
            "tight_bbox": tight_box,
        })

    print(f"  Step 1: {len(valid)} valid masks from {len(pairs)} raw")

    # ── Step 2: Dedup by mask IoU ────────────────────────────────────────
    # Sort by mask area descending — larger masks are "primary"
    valid = sorted(valid, key=lambda v: v["mask_area"], reverse=True)

    kept = []
    skipped_log = []
    for cand in valid:
        skip = False
        for prev in kept:
            iou = _mask_iou(cand["mask_bool"], prev["mask_bool"])
            if iou >= iou_dedup_threshold:
                skip = True
                skipped_log.append(
                    f"#{cand['mask_index']}(dup of #{prev['mask_index']}, "
                    f"iou={iou:.2f})")
                break
        if not skip:
            kept.append(cand)

    if skipped_log:
        print(f"  Step 2: dedup removed {len(skipped_log)}: "
              + ", ".join(skipped_log))
    print(f"  Step 2: {len(kept)} unique segments after dedup")

    # ── Step 3-4: Masked crop + aspect-preserving resize ─────────────────
    results = []
    for item in kept:
        mask_bool = item["mask_bool"]
        tight_box = item["tight_bbox"]

        # Add margin, clamped to image bounds
        margin_box = _add_margin(tight_box, (h, w),
                                 margin_frac=margin_frac,
                                 min_margin_px=min_margin_px)
        mx1, my1, mx2, my2 = margin_box

        # Crop image and mask
        crop_img = image_np[my1:my2, mx1:mx2].copy()
        crop_mask = mask_bool[my1:my2, mx1:mx2]

        # Apply mask: segment pixels keep original, rest → bg_color
        crop_img[~crop_mask] = bg

        # Aspect-preserving resize with white padding
        patch = _resize_aspect_pad(crop_img, (patch_h, patch_w),
                                   pad_value=bg_color[0])

        # Compute coverage (segment pixels / total patch pixels)
        # Resize the mask too to compute actual coverage in output
        mask_crop_pil = Image.fromarray(crop_mask.astype(np.uint8) * 255)
        src_ch, src_cw = crop_img.shape[:2]
        scale = min(patch_w / float(max(1, src_cw)),
                    patch_h / float(max(1, src_ch)))
        rw = max(1, int(round(src_cw * scale)))
        rh = max(1, int(round(src_ch * scale)))
        resized_mask = np.array(
            mask_crop_pil.resize((rw, rh), resample=Image.NEAREST)
        ) > 127
        coverage = float(resized_mask.sum()) / float(max(1, patch_h * patch_w))

        results.append({
            "mask_index": item["mask_index"],
            "bbox_xywh": item["bbox_xywh"],
            "tight_bbox": tight_box,
            "mask_area": item["mask_area"],
            "margin_bbox": margin_box,
            "segment_coverage": coverage,
            "patch": patch,
        })

    # Sort by mask index for deterministic output
    results = sorted(results, key=lambda r: r["mask_index"])
    print(f"  Step 3-4: {len(results)} final patches")
    return results


# ─── Visualization ───────────────────────────────────────────────────────────

def show_all_segmented_outputs(
    images,
    pairs_per_img,
    alpha=0.45,
    cols=6,
    seed=42,
    patch_shape=(200, 200),
    iou_dedup_threshold=0.80,
    bg_color=(255, 255, 255),
):
    """
    For each image:
      1) Full-image combined overlay (all masks with random colors + bboxes).
      2) Grid of clean masked concept patches from build_segment_patches.
    """
    if len(images) != len(pairs_per_img):
        raise ValueError("images and pairs_per_img must have the same length")

    rng = np.random.default_rng(seed)

    for img_idx, (image, pairs) in enumerate(zip(images, pairs_per_img), start=1):
        print(f"Image {img_idx}: raw masks={len(pairs)}")
        if len(pairs) == 0:
            continue

        # 1) Full-image combined overlay
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        for bbox, mask in pairs:
            if mask is None:
                continue
            color = rng.random(3)
            plt.imshow(_mask_overlay(np.asarray(mask).astype(bool), color, alpha=alpha))
            x, y, w_box, h_box = bbox
            plt.gca().add_patch(
                plt.Rectangle((x, y), w_box, h_box, fill=False,
                               edgecolor="yellow", linewidth=1.2)
            )
        plt.title(f"Image {img_idx}: combined overlay ({len(pairs)} masks)")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        # 2) Clean masked concept patches
        patches = build_segment_patches(
            image, pairs,
            patch_shape=patch_shape,
            iou_dedup_threshold=iou_dedup_threshold,
            bg_color=bg_color,
        )
        print(f"Image {img_idx}: {len(patches)} concept patches")
        if len(patches) == 0:
            continue

        n = len(patches)
        rows = math.ceil(n / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
        axes = np.array(axes).reshape(rows, cols)

        for idx in range(rows * cols):
            ax = axes[idx // cols, idx % cols]
            ax.axis("off")
            if idx >= n:
                continue
            p = patches[idx]
            ax.imshow(p["patch"])
            ax.set_title(
                f"#{p['mask_index']}  area={p['mask_area']:,}  "
                f"cov={p['segment_coverage']:.2f}",
                fontsize=9,
            )

        plt.suptitle(
            f"Image {img_idx}: masked concept patches {patch_shape} "
            f"(white bg, {len(patches)} segments)",
            y=1.02,
        )
        plt.tight_layout()
        plt.show()


# Plotting

In [ ]:
# Show all segmented outputs for each processed image
show_all_segmented_outputs(
    images_for_auto,
    auto_pairs_per_img,
    alpha=0.45,
    cols=6,
    seed=7,
 )

In [ ]:
# Bounding boxes summary (top 10 per image)
for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    for idx, (bbox, mask) in enumerate(pairs[:10], start=1):
        print(f"img{image_idx} mask {idx:02d}: bbox={bbox}, area={int(mask.sum())}")

In [ ]:
# Image-level mask count check
for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    print(f"Image {image_idx}: {len(pairs)} mask(s)")

In [ ]:
# Display first mask bbox for each image (if available)
for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    if len(pairs) == 0:
        print(f"Image {image_idx}: no masks")
    else:
        bbox, mask = pairs[0]
        print(f"Image {image_idx}: first mask bbox={bbox}, area={int(mask.sum())}")

In [ ]:
# Total segmented outputs across all images
total_masks = sum(len(pairs) for pairs in auto_pairs_per_img)
print(f"Total segmented outputs: {total_masks}")

In [ ]:
# Per-image average mask area
for image_idx, pairs in enumerate(auto_pairs_per_img, start=1):
    areas = [int(mask.sum()) for _, mask in pairs if mask is not None]
    avg_area = float(np.mean(areas)) if areas else 0.0
    print(f"Image {image_idx}: avg mask area={avg_area:.1f}")

In [ ]:
# Final prompt-free segmentation summary
print({
    "num_images": len(images_for_auto),
    "masks_per_image_target": masks_per_image,
    "actual_masks_per_image": [len(pairs) for pairs in auto_pairs_per_img],
    "points_per_side": points_per_side,
    "topn": auto_topn,
    "min_mask_area": min_mask_area,
})